# Phase 3 - Consolidated Classification Notebook

This notebook is the runnable consolidation of the repaired Phase 3 pipeline:

1. Runtime path setup and preflight checks
2. Dataset root resolution and metadata indexing
3. Deterministic sampled dataloaders
4. Model and trainer initialization
5. Smoke training run (short)
6. Row-level decision engine demo
7. Run artifact export

The flow is notebook-first and avoids legacy Trial3 hardcoded paths.

## 1. Runtime Setup

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

# Robust project root discovery for local and Colab-style runtimes.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "OMR").exists() and (p / "Unified_Datasets").exists()), cwd)

PHASE3_SRC = PROJECT_ROOT / "OMR" / "Phase_3_Classification" / "src"
if str(PHASE3_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE3_SRC))

run_stamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
RUN_DIR = PROJECT_ROOT / "OMR" / "runs" / run_stamp
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"PHASE3_SRC   : {PHASE3_SRC}")
print(f"RUN_DIR      : {RUN_DIR}")
print(f"SRC exists   : {PHASE3_SRC.exists()}")

In [ ]:
import json
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd
import torch

from bubble_classifier import BubbleClassifier
from cnn_models import AscendingCNN, DiamondCNN
from dataset import build_dataset_index, create_dataloaders, resolve_phase3_dataset_root
from scoring import RelativeRowDecisionEngine
from trainer import ClassificationTrainer, TrainerConfig

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {DEVICE}")

## 2. Dataset Preflight and Indexing

This section resolves dataset roots without legacy hardcoded Trial3 paths and writes a metadata index for reproducibility.

In [ ]:
resolution = resolve_phase3_dataset_root(PROJECT_ROOT)
dataset_root = resolution.dataset_root

phase2_cropped = PROJECT_ROOT / "Unified_Datasets" / "Phase_2_Cropped"
phase3_ready = PROJECT_ROOT / "Unified_Datasets" / "Phase_3_Ready"
manual_labeled = PROJECT_ROOT / "Unified_Datasets" / "manual_labeled"

counts = {}
for name, path in {
    "phase2_cropped": phase2_cropped,
    "phase3_ready": phase3_ready,
    "manual_labeled": manual_labeled,
}.items():
    if path.exists():
        counts[name] = sum(1 for p in path.rglob("*") if p.is_file())
    else:
        counts[name] = 0

print(f"Resolved dataset root : {dataset_root} (source={resolution.source_name})")
print("Dataset file counts   :", counts)

handoff_blocked = counts["phase3_ready"] == 0 and counts["phase2_cropped"] == 0
if handoff_blocked:
    print("[WARN] handoff-blocked: Phase_2_Cropped and Phase_3_Ready are empty.")
    print("[INFO] Falling back to manual_labeled for smoke training.")

## 3. Build Metadata Index and Dataloaders

In [ ]:
def resize_transform(image):
    # Fixed spatial size avoids dataloader stacking errors.
    return cv2.resize(image, (64, 64), interpolation=cv2.INTER_AREA)

index_path = dataset_root / "dataset_index.csv"
index_df = build_dataset_index(dataset_root=dataset_root, index_output_path=index_path)

print(f"Index path     : {index_path}")
print(f"Indexed rows   : {len(index_df):,}")
print("Label counts:")
print(index_df["label"].value_counts(dropna=False))
print("Split counts:")
print(index_df["split"].value_counts(dropna=False))
index_df.head(5)

In [ ]:
bundle = create_dataloaders(
    project_root=PROJECT_ROOT,
    batch_size=32,
    num_workers=0,
    val_split_ratio=0.2,
    random_seed=SEED,
    sample_limit=2000,
    auto_sample_threshold=5000,
    transform_train=resize_transform,
    transform_valid=resize_transform,
    rebuild_index=False,
 )

train_loader = bundle.train_loader
valid_loader = bundle.valid_loader
class_to_idx = bundle.class_to_idx

print(f"Bundle dataset root : {bundle.dataset_root}")
print(f"Bundle index path   : {bundle.index_path}")
print(f"Sample mode         : {bundle.sample_mode}")
print(f"Class map           : {class_to_idx}")
print(f"Train batches       : {len(train_loader)}")
print(f"Valid batches       : {len(valid_loader)}")

## 4. Model Initialization and Smoke Training

In [ ]:
idx_to_class = {idx: name for name, idx in class_to_idx.items()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]

num_classes = len(class_names)
model = AscendingCNN(num_classes=num_classes) if num_classes <= 3 else DiamondCNN(num_classes=num_classes)

trainer_config = TrainerConfig(
    epochs=1,
    learning_rate=1e-3,
    weight_decay=1e-4,
    early_stopping_patience=2,
    early_stopping_min_delta=1e-4,
    output_dir=str(RUN_DIR / "models"),
    monitor_metric="val_f1",
    device="auto",
)

trainer = ClassificationTrainer(
    model=model,
    class_names=class_names,
    config=trainer_config,
)

print(f"Classes      : {class_names}")
print(f"Model        : {model.__class__.__name__}")
print(f"Output dir   : {trainer.output_dir}")

In [ ]:
summary = trainer.train(
    train_loader=train_loader,
    valid_loader=valid_loader,
)

print("Training summary:")
print(json.dumps({
    "best_epoch": summary["best_epoch"],
    "best_metric": summary["best_metric"],
    "monitor_metric": summary["monitor_metric"],
    "epochs_ran": summary["epochs_ran"],
    "best_checkpoint": summary["best_checkpoint"],
}, indent=2))

## 5. Row-Level Decision Engine Demo

In [ ]:
engine = RelativeRowDecisionEngine(
    min_fill_prob=0.30,
    tie_threshold=0.15,
    multi_mark_threshold=0.50,
    review_band=0.06,
)

labels = ["A", "B", "C", "D", "E"]
scenarios = [
    ("clear_C", [0.05, 0.08, 0.82, 0.03, 0.02]),
    ("tie_A_B", [0.48, 0.46, 0.02, 0.02, 0.02]),
    ("blank", [0.10, 0.12, 0.08, 0.09, 0.11]),
    ("multi_mark", [0.62, 0.60, 0.08, 0.03, 0.02]),
]

rows = []
for name, probs in scenarios:
    d = engine.decide_row(probs, labels)
    rows.append({
        "scenario": name,
        "selected": d.selected,
        "status": d.status,
        "is_blank": d.is_blank,
        "is_tie": d.is_tie,
        "is_multi_mark": d.is_multi_mark,
        "needs_review": d.needs_review,
        "confidence": round(d.confidence, 4),
        "margin_top2": round(d.margin_top2, 4),
    })

pd.DataFrame(rows)

## 6. Optional Phase 3 Purge Step (Dry Run)

Use this to inspect solidity/darkness cleaning decisions before producing Phase_3_Ready assets.

In [ ]:
from purge_data import PurgeConfig, run_purge

purge_report = RUN_DIR / "purge_report.csv"
purge_output = PROJECT_ROOT / "Unified_Datasets" / "Phase_3_Ready"
purge_rejects = PROJECT_ROOT / "Unified_Datasets" / "Phase_3_Rejects"
purge_source = PROJECT_ROOT / "Unified_Datasets" / "Phase_2_Cropped"

if not purge_source.exists() or not any(p.is_file() for p in purge_source.rglob("*")):
    purge_source = PROJECT_ROOT / "Unified_Datasets" / "manual_labeled"

config = PurgeConfig(
    source_root=purge_source,
    output_root=purge_output,
    reject_root=purge_rejects,
    report_csv=purge_report,
    solidity_threshold=0.72,
    darkness_threshold=0.06,
    uncertainty_margin=0.05,
    dry_run=True,
)

purge_counts = run_purge(config)
print("Dry-run purge counts:", purge_counts)
print(f"Purge report: {purge_report}")

## 7. Export Run Summary Dossier

In [ ]:
run_summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(dataset_root),
    "dataset_source": resolution.source_name,
    "dataset_counts": counts,
    "handoff_blocked": handoff_blocked,
    "index_path": str(index_path),
    "sample_mode": bundle.sample_mode,
    "class_to_idx": class_to_idx,
    "trainer_output_dir": str(trainer.output_dir),
    "training_summary_path": str(trainer.summary_path),
    "best_checkpoint": str(trainer.best_checkpoint_path),
    "purge_report_csv": str(purge_report),
    "purge_counts": purge_counts,
}

summary_path = RUN_DIR / "run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

print(f"Run summary saved: {summary_path}")
print(json.dumps(run_summary, indent=2))

## Notebook Consolidation Result

This notebook now runs as a consolidated Phase 3 workflow using the repaired source modules:

- dataset indexing and deterministic dataloaders
- Ascending or Diamond CNN training smoke test
- row-level decision arbitration with review flags
- optional purge-data dry-run
- run dossier export to `OMR/runs/<timestamp>/run_summary.json`

If you need a longer training run, increase `epochs` and `sample_limit` in the training cells.